# 05 - Position-restricted activation patching

Causal localization. Cache the residual stream from a **clean** (14pt) run and
splice it into a **corrupt** (5pt) run, one layer at a time, and measure how much
clean behaviour is recovered.

## Two design constraints, both load-bearing

**1. Patch a restricted token span, never all positions.** An unrestricted
version recovers 1.000 at *every* layer including layer 0 - degenerate by
construction, since replacing layer 0's full output means every subsequent layer
computes on clean values. That is not a finding, it is a tautology. We patch
either the vision-token span or the single answer position.

**2. Clean and corrupt must align.** Qwen2.5-VL uses dynamic resolution, so token
counts key off image dimensions. With a fixed Typst page box both fonts render to
the same canvas and produce identical token counts - but this must be *verified*,
not assumed, and the renders must also actually differ (check pixel diff, not
just the font setting).

The self-validation cell below asserts on both. A degenerate design fails in
minutes instead of after three hours.

Produces `results/patching/`.

> Reconstructed from session transcripts; outputs not embedded.

## Setup

Clone the repo, install deps, load Qwen2.5-VL-7B in bf16 across 2x T4.

**Check Accelerator = GPU T4 x2 before running.** A Kaggle batch job inherits
`None` silently and runs at ~1200 s/item on CPU. The assert below catches it.

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
import os, sys
if not os.path.isdir("/kaggle/working/algoverse"):
    !git clone https://{token}@github.com/bryantran21/algoverse.git /kaggle/working/algoverse
sys.path.insert(0, "/kaggle/working/algoverse"); os.chdir("/kaggle/working/algoverse")
!git config user.email "bryantran21@gmail.com"
!git config user.name  "bryantran21"

import subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.49.0',
    'accelerate>=0.34.0','datasets','qwen-vl-utils','typst','Pillow',
    'scikit-learn','matplotlib','tqdm'], check=True)

import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count(), "devices")
assert torch.cuda.is_available(), "NO GPU - set Accelerator to T4 x2 before running"

import config
config.DEVICE_MAP = "auto"        # 2-GPU full precision
config.LOAD_IN_4BIT = False       # 4-bit perturbs activations - never use for interp
from src.inference import load_vl_model
model, processor = load_vl_model()
print("READY", flush=True)

## Alignment check

Confirm 5pt and 14pt produce identical sequence lengths and identical
`<|image_pad|>` counts, and that the two renders are not byte-identical images.

In [ ]:
import config, numpy as np, torch
from src.inference import _messages_for, _flatten_images

vis_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
sets = __import__("pickle").load(open("results/sets/sets_n2000.pkl","rb"))
fail_all, ctrl_all = sets["fail_all"], sets["ctrl_all"]

def tok_profile(item, fs):
    config.RENDER["font_size_pt"] = fs
    msgs, images = _messages_for(item, "image")
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs = _flatten_images([images])
    inp = processor(text=[text], images=imgs, return_tensors="pt")
    return inp["input_ids"].shape[1], int((inp["input_ids"] == vis_id).sum()), imgs[0].size

for it in fail_all[:3]:
    l5, v5, sz5 = tok_profile(it, 5.0)
    l14, v14, sz14 = tok_profile(it, 14.0)
    print(f"item {it['item_id']}: 5pt len={l5} vis={v5} canvas={sz5} | "
          f"14pt len={l14} vis={v14} canvas={sz14} | "
          f"{'ALIGNED' if (l5==l14 and v5==v14) else 'MISALIGNED'}")

# renders must differ - identical images would mean font size does nothing
it = fail_all[0]
config.RENDER["font_size_pt"] = 5.0
a = np.array(_flatten_images([_messages_for(it,"image")[1]])[0].convert("L"))
config.RENDER["font_size_pt"] = 14.0
b = np.array(_flatten_images([_messages_for(it,"image")[1]])[0].convert("L"))
print("identical:", np.array_equal(a,b), " mean abs diff:", np.abs(a.astype(int)-b.astype(int)).mean().round(2))
config.RENDER["font_size_pt"] = 5.0

## Patching machinery + self-validation

`patched()` replaces layer L's output **only at positions where `pos_mask` is
True**, leaving every other position at its corrupt value.

The validation probe checks three layers under both masks. If everything returns
1.000, the design is degenerate and the assert aborts before the long run.

In [ ]:
import pickle, os, time
os.makedirs("results/patching", exist_ok=True)
YES, NO = 9454, 2753
layers = model.language_model.layers if hasattr(model, "language_model") \
         else model.model.language_model.layers
N_LAYERS = len(layers)

def build(item, fs):
    config.RENDER["font_size_pt"] = fs
    msgs, images = _messages_for(item, "image")
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs = _flatten_images([images])
    return processor(text=[text], images=imgs, return_tensors="pt").to(model.device)

@torch.no_grad()
def cache_clean(inputs):
    store, hooks = {}, []
    for i, layer in enumerate(layers):
        def mk(i):
            def hook(mod, inp, out):
                store[i] = (out[0] if isinstance(out, tuple) else out).detach().clone()
            return hook
        hooks.append(layer.register_forward_hook(mk(i)))
    model(**inputs, use_cache=False)
    for h in hooks: h.remove()
    return store

@torch.no_grad()
def patched(inputs, store, L, pos_mask):
    def hook(mod, inp, out):
        tup = isinstance(out, tuple)
        h = (out[0] if tup else out).clone()
        h[:, pos_mask, :] = store[L][:, pos_mask, :]
        return ((h,) + out[1:]) if tup else h
    hk = layers[L].register_forward_hook(hook)
    try:
        lg = model(**inputs, use_cache=False).logits[0, -1].float()
    finally:
        hk.remove()
    return (lg[YES] - lg[NO]).item()

@torch.no_grad()
def plain(inputs):
    lg = model(**inputs, use_cache=False).logits[0, -1].float()
    return (lg[YES] - lg[NO]).item()

# ---- self-validation: abort if the design is degenerate ----
it = fail_all[0]
ci, xi = build(it, 14.0), build(it, 5.0)
ids = xi["input_ids"][0]; n = ids.shape[0]
vis_mask = (ids == vis_id).cpu().numpy()
ans_mask = np.zeros(n, dtype=bool); ans_mask[-1] = True

d_c, d_x = plain(ci), plain(xi)
store = cache_clean(ci)
print(f"clean {d_c:+.3f}  corrupt {d_x:+.3f}  seq {n}  vision {vis_mask.sum()}", flush=True)

probe = {}
for name, mask in [("VISION", vis_mask), ("ANSWER", ans_mask)]:
    vals = [(patched(xi, store, L, mask) - d_x) / (d_c - d_x) for L in (0, 14, 27)]
    probe[name] = vals
    print(f"{name}: L0 {vals[0]:+.3f}  L14 {vals[1]:+.3f}  L27 {vals[2]:+.3f}", flush=True)
del store; torch.cuda.empty_cache()

flat = all(abs(v - 1.0) < 1e-3 for vs in probe.values() for v in vs)
assert not flat, "DEGENERATE: masked patching still recovers 1.0 everywhere - design is wrong"
print("validation OK", flush=True)

## Full patching run

For each item, sweep all 28 layers under both masks. Recovery is normalized so
0 = corrupt behaviour and 1 = clean behaviour.

Items are skipped when clean and corrupt give identical logit differences - there
is nothing to recover.

~5 min/item. Checkpoints every 5.

In [ ]:
ITEMS = fail_all[:30]
MASKS = ["vision", "answer"]
results = {m: [] for m in MASKS}
meta, t0 = [], time.time()

for k, it in enumerate(ITEMS):
    ci, xi = build(it, 14.0), build(it, 5.0)
    if ci["input_ids"].shape[1] != xi["input_ids"].shape[1]:
        print(f"skip {it['item_id']} (misaligned)", flush=True); continue
    d_c, d_x = plain(ci), plain(xi)
    if abs(d_c - d_x) < 1e-6:
        print(f"skip {it['item_id']} (no effect)", flush=True); continue

    ids = xi["input_ids"][0]; n = ids.shape[0]
    masks = {"vision": (ids == vis_id).cpu().numpy(),
             "answer": np.zeros(n, dtype=bool)}
    masks["answer"][-1] = True

    store = cache_clean(ci)
    for m in MASKS:
        results[m].append([(patched(xi, store, L, masks[m]) - d_x) / (d_c - d_x)
                           for L in range(N_LAYERS)])
    del store; torch.cuda.empty_cache()
    meta.append(dict(item_id=it["item_id"], gold=it["gold"], d_clean=d_c, d_corrupt=d_x))

    if (k+1) % 5 == 0:
        pickle.dump({"results": results, "meta": meta},
                    open("results/patching/patch_masked.pkl","wb"))
        print(f"{k+1}/{len(ITEMS)}  ({(time.time()-t0)/60:.1f} min)", flush=True)

pickle.dump({"results": results, "meta": meta},
            open("results/patching/patch_masked.pkl","wb"))

for m in MASKS:
    A = np.array(results[m])
    print(f"\n{m}: n={len(A)}  mean by layer:", np.round(A.mean(axis=0), 3), flush=True)
config.RENDER["font_size_pt"] = 5.0

### Patching figure

The two curves cross at layers 18-20. Vision-token patching stops working there;
answer-position patching starts working there. That window is where information
moves out of the visual representation and into the decision - and it is why the
answer only becomes decodable at layer 27.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4.5))
cols = {"vision": "#1E2761", "answer": "#C0392B"}
for m in MASKS:
    A = np.array(results[m]); mu = A.mean(axis=0)
    rng = np.random.default_rng(0)
    bs = A[rng.integers(0, len(A), size=(1000, len(A)))].mean(axis=1)
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    L = np.arange(A.shape[1])
    ax.plot(L, mu, "o-", color=cols[m], ms=4, label=f"{m} tokens")
    ax.fill_between(L, lo, hi, color=cols[m], alpha=0.15)
ax.axhline(0, color="gray", ls=":", label="corrupt (5pt)")
ax.axhline(1, color="black", ls=":", label="clean (14pt)")
ax.set_xlabel("patched layer"); ax.set_ylabel("normalized recovery")
ax.set_title(f"Position-restricted activation patching (n={len(meta)})")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig("results/patching/patch_masked.png", dpi=150); plt.show()

np.savez("results/patching/patch_masked.npz",
         vision=np.array(results["vision"]), answer=np.array(results["answer"]))

In [ ]:
!git add -A && git commit -m "05: position-restricted activation patching (vision vs answer span)" && git push origin master